# 03 — Pipeline + Cross-validation
ColumnTransformer → 3 models compared → feature importances

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_validate
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')

with open('../data/splits.pkl', 'rb') as f:
    X_train, X_test, y_train, y_test, NUM, ORD, NOM = pickle.load(f)

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')

## Build ColumnTransformer

In [ ]:
# Keep only columns present in the data
num_cols = [c for c in NUM if c in X_train.columns]
ord_cols = [c for c in ORD if c in X_train.columns]
nom_cols = [c for c in NOM if c in X_train.columns]

ordinal_categories = [ORD[c] for c in ord_cols]

numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])

ordinal_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(
        categories=ordinal_categories,
        handle_unknown='use_encoded_value',
        unknown_value=-1,
    )),
])

nominal_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipe, num_cols),
    ('ord', ordinal_pipe, ord_cols),
    ('nom', nominal_pipe, nom_cols),
])

print('Preprocessor built.')
print(f'  Numeric  : {len(num_cols)} cols')
print(f'  Ordinal  : {len(ord_cols)} cols')
print(f'  Nominal  : {len(nom_cols)} cols')

## Cross-validate 3 Models

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge (alpha=10)':  Ridge(alpha=10),
    'Random Forest':     RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
}

cv_results = {}
for name, model in models.items():
    pipe = Pipeline([('pre', preprocessor), ('model', model)])
    scores = cross_validate(
        pipe, X_train, y_train,
        cv=5,
        scoring=['neg_root_mean_squared_error', 'neg_mean_absolute_error', 'r2'],
        return_train_score=False,
        n_jobs=-1,
    )
    cv_results[name] = {
        'RMSE': -scores['test_neg_root_mean_squared_error'].mean(),
        'RMSE_std': scores['test_neg_root_mean_squared_error'].std(),
        'MAE':  -scores['test_neg_mean_absolute_error'].mean(),
        'R2':   scores['test_r2'].mean(),
    }
    print(f"{name:25s}  RMSE={cv_results[name]['RMSE']:.4f}  R2={cv_results[name]['R2']:.4f}")

results_df = pd.DataFrame(cv_results).T
print('\nFull CV Results:')
print(results_df.to_string())

## Model Comparison Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

names = list(cv_results.keys())
rmses = [cv_results[n]['RMSE'] for n in names]
r2s   = [cv_results[n]['R2']   for n in names]

axes[0].bar(names, rmses, color=['steelblue', 'coral', 'mediumseagreen'])
axes[0].set_title('CV RMSE (lower is better)')
axes[0].set_ylabel('RMSE (log scale)')
axes[0].tick_params(axis='x', rotation=15)

axes[1].bar(names, r2s, color=['steelblue', 'coral', 'mediumseagreen'])
axes[1].set_title('CV R² (higher is better)')
axes[1].set_ylabel('R²')
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('../data/model_comparison.png', bbox_inches='tight')
plt.show()

## Feature Importances (Random Forest)

In [ ]:
rf_pipe = Pipeline([('pre', preprocessor), ('model', RandomForestRegressor(n_estimators=200, random_state=42))])
rf_pipe.fit(X_train, y_train)

# Get feature names from ColumnTransformer
ohe_names = rf_pipe.named_steps['pre'].named_transformers_['nom']['encoder'].get_feature_names_out(nom_cols)
feature_names = num_cols + ord_cols + list(ohe_names)

importances = rf_pipe.named_steps['model'].feature_importances_
imp_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
imp_df = imp_df.sort_values('importance', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(8, 7))
sns.barplot(data=imp_df, y='feature', x='importance', ax=ax, palette='Blues_r')
ax.set_title('Top 20 Feature Importances — Random Forest')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('../data/feature_importances.png', bbox_inches='tight')
plt.show()

# Save pipeline and results for notebook 04
with open('../data/rf_pipe.pkl', 'wb') as f:
    pickle.dump(rf_pipe, f)
with open('../data/cv_results.pkl', 'wb') as f:
    pickle.dump((cv_results, preprocessor), f)
print('Saved rf_pipe.pkl and cv_results.pkl')